# 13. The bulk column of the transfer matrix

ITransverse builds the evolution operator on three sites and repeats the middle tensor as the
column of the rotated network. For a nearest-neighbour model the middle site of a 3-site chain is
a genuine bulk site, so this is exact. For our NNN model it is not: that site has no site+2 to
couple to, so the extracted column misses a memory channel and carries temporal physical dimension
7 where the true bulk tensor carries 13. Every production result at $p\neq0$ so far was computed
on the 7-dimensional column.

`bulk_fwtmpoblocks` in `src/transverse_tools.jl` builds the corrected column from the middle
tensor of a 5-site operator, reached through `column=:bulk5` in `build_tmpo`, `build_alcaraz_tmpo`,
`compute_entropies` and `ksector_signs`. This notebook validates it before any cluster run: the
operator-level exactness of the extractor, the $p=0$ null, the physical echo against an exact
Schr\"odinger calculation, the sector structure on the corrected column, and a production smoke
test.

## The extractor at operator level

Tiling a genuine bulk tensor over $N$ sites must reproduce the honest $N$-site evolution operator
exactly; tiling the 3-site middle tensor cannot once the NNN term is on. The cell builds both
columns at each coupling, tiles them over six sites, and compares against the honest six-site MPO
contracted densely. The corrected column must sit at machine zero for every $p$ and both kernels;
the legacy one is expected to fail at every $p\neq0$.

In [1]:
include("../src/thesislib.jl")
using Printf, LinearAlgebra
ITensors.disable_warn_order()

function dense_mpo_matrix(tensors, sites)
    c = tensors[1]
    for i in 2:length(tensors); c = c * tensors[i]; end
    cc = combiner(sites...); rc = combiner(prime.(sites)...)
    d = rc * c * cc
    return Matrix(d, combinedind(rc), combinedind(cc))
end

# keep the edge tensors of the m-site build, tile its middle tensor over the interior, and
# compare with the honest N-site operator
function tiling_error(p, alg, m; N=6, dt=0.1)
    sites = [Index(2, "S=1/2,Site,n=$i") for i in 1:N]
    Ufull = expH_alcaraz(sites, 1.0, p; dt=dt, mpo_alg=alg)
    A = dense_mpo_matrix([Ufull[i] for i in 1:N], sites)

    sm = sites[1:m]
    Um = expH_alcaraz(sm, 1.0, p; dt=dt, mpo_alg=alg)
    L = linkinds(Um)
    mid = (m + 1) ÷ 2
    dim(L[mid-1]) == dim(L[mid]) || return NaN
    nedge = mid - 1
    ntile = N - 2 * nedge

    T = ITensor[]
    for i in 1:nedge
        push!(T, replaceinds(Um[i], (sm[i], sm[i]'), (sites[i], sites[i]')))
    end
    lcur = L[mid-1]
    for j in 1:ntile
        n = nedge + j
        rnew = j == ntile ? L[mid] : sim(L[mid]; tags="splice$j")
        push!(T, replaceinds(Um[mid], (L[mid-1], L[mid], sm[mid], sm[mid]'),
                             (lcur, rnew, sites[n], sites[n]')))
        lcur = rnew
    end
    for (k, i) in enumerate(mid+1:m)
        n = nedge + ntile + k
        push!(T, replaceinds(Um[i], (sm[i], sm[i]'), (sites[n], sites[n]')))
    end
    B = dense_mpo_matrix(T, sites)
    return norm(A - B) / norm(A)
end

@printf("%-6s %-5s %-14s %-14s\n", "p", "alg", "legacy (m=3)", "bulk (m=5)")
for p in (0.0, 0.1, 0.3, 0.5), alg in ("VD2", "WII")
    @printf("%-6.1f %-5s %-14.3e %-14.3e\n", p, alg, tiling_error(p, alg, 3), tiling_error(p, alg, 5))
end

p      alg   legacy (m=3)   bulk (m=5)    
0.0    VD2   0.000e+00      0.000e+00     


0.0    WII   0.000e+00      0.000e+00     
0.1    VD2   3.422e-01      0.000e+00     


0.1    WII   3.404e-01      0.000e+00     
0.3    VD2   3.450e-01      0.000e+00     


0.3    WII   3.422e-01      0.000e+00     
0.5    VD2   3.522e-01      0.000e+00     
0.5    WII   3.473e-01      0.000e+00     


## The null at $p=0$ and the dimensions at $p\neq0$

At $p=0$ the model is nearest-neighbour, so the two extractions must produce the identical
transfer matrix, tensor by tensor. At $p\neq0$ only the dimensions are compared here; the physics
comparison follows below.

In [2]:
for p in (0.0, 0.1, 0.3, 0.5)
    m3, _ = build_alcaraz_tmpo(1.0; p=p, nbeta=4, column=:legacy3)
    m5, _ = build_alcaraz_tmpo(1.0; p=p, nbeta=4, column=:bulk5)
    d3, d5 = dim(siteind(m3, 2)), dim(siteind(m5, 2))
    if p == 0.0
        delta = norm(Array(m3[3], inds(m3[3])...) - Array(m5[3], inds(m5[3])...))
        @printf("p=%.1f  site dim %d vs %d   |legacy - bulk| on the bulk tensor = %.2e\n", p, d3, d5, delta)
    else
        @printf("p=%.1f  site dim %d vs %d\n", p, d3, d5)
    end
end

[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
[ Info: Tensor symmetric (dim=2|id=753|"S=1/2,Site") <-> (dim=2|id=753|"S=1/2,Site")'


[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=3|id=425|"Link,l=1") <-> (dim=3|id=636|"Link,l=2"), normdiff = 0.944688073151956
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
[ Info: Tensor symmetric (dim=2|id=360|"S=1/2,Site") <-> (dim=2|id=360|"S=1/2,Site")'


[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=3|id=159|"Link,l=1") <-> (dim=3|id=761|"Link,l=2"), normdiff = 0.10493384159131464
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
[ Info: Tensor symmetric (dim=2|id=304|"S=1/2,Site") <-> (dim=2|id=304|"S=1/2,Site")'


[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=3|id=877|"Link,l=1") <-> (dim=3|id=648|"Link,l=2"), normdiff = 0.944688073151956
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
[ Info: Tensor symmetric (dim=2|id=4|"S=1/2,Site") <-> (dim=2|id=4|"S=1/2,Site")'
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=3|id=216|"Link,l=1") <-> (dim=3|id=735|"Link,l=2"), normdiff = 0.10493384159131464
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


p=0.0  site dim 3 vs 3   |legacy - bulk| on the bulk tensor = 0.00e+00


[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=605|"S=1/2,Site") <-> (dim=2|id=605|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=946|"Link,l=1") <-> (dim=7|id=118|"Link,l=2"), normdiff = 1.1566412865695994
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=185|"S=1/2,Site") <-> (dim=2|id=185|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=155|"Link,l=1") <-> (

p=0.1  site dim 7 vs 13


p=0.3  site dim 7 vs 13
p=0.5  site dim 7 vs 13


[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=455|"S=1/2,Site") <-> (dim=2|id=455|"S=1/2,Site")', normdiff = 0.22953505077375283
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=597|"Link,l=1") <-> (dim=7|id=713|"Link,l=2"), normdiff = 1.222384963003563
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=698|"S=1/2,Site") <-> (dim=2|id=698|"S=1/2,Site")', normdiff = 0.22953505077375283
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=135|"Link,l=1") <-> 

## The echo against an exact Schr\"odinger calculation

This is the decisive physical test, because it needs no transverse machinery at all. The regulated
echo $A_N(T)=\bra{X+}e^{-\beta_0 H}e^{-iHT}e^{-\beta_0 H}\ket{X+}$ of a finite open chain is
computed by Krylov evolution of the dense state vector, exactly in time. Its intensive rate
$\ell_N=-\log|A_N|/N$ approaches $-\log|\mu_0|$ as $N\to\infty$, with a $1/N$ boundary
correction, so fitting $\ell_N=\ell_\infty+a/N$ over three chain sizes gives an
extrapolated rate to compare against each column's leading eigenvalue.

The $p=0$ row calibrates the error budget: both columns are identical there, so whatever
$|\ell_\infty+\log|\mu_0||$ remains at $p=0$ measures the Trotter error of the $\delta t=0.1$
operator plus the extrapolation residual. A column is validated if its deviation at $p\neq0$
stays at that floor, and refuted if it sits well above it.

In [3]:
using KrylovKit, SparseArrays, JLD2

# open-chain H as a sparse matrix, same convention as Eq. (1)
function sparse_open_H(N, lambda, p)
    rows = Int[]; cols = Int[]; vals = Float64[]
    for s in 1:2^N
        bits = digits(s - 1, base=2, pad=N)
        spin(i) = 1 - 2 * bits[i]
        diag = 0.0
        for i in 1:N-1; diag -= spin(i) * spin(i+1); end
        for i in 1:N-2; diag -= p * spin(i) * spin(i+2); end
        push!(rows, s); push!(cols, s); push!(vals, diag)
        for i in 1:N
            flipped = copy(bits); flipped[i] = 1 - flipped[i]
            push!(rows, 1 + sum(flipped[k] * 2^(k-1) for k in 1:N)); push!(cols, s); push!(vals, -lambda)
        end
        for i in 1:N-1
            flipped = copy(bits); flipped[i] = 1 - flipped[i]; flipped[i+1] = 1 - flipped[i+1]
            push!(rows, 1 + sum(flipped[k] * 2^(k-1) for k in 1:N)); push!(cols, s); push!(vals, -p * lambda)
        end
    end
    return sparse(rows, cols, vals, 2^N, 2^N)
end

# regulated echo rate of the finite chain, exact in time
function krylov_rate(N, p, T; beta0=0.2)
    H = sparse_open_H(N, 1.0, p)
    psi = fill(ComplexF64(1 / sqrt(2.0^N)), 2^N)          # |X+>^N in the z basis
    psi, _ = exponentiate(x -> H * x, -beta0, psi; tol=1e-10, ishermitian=true)
    phi = copy(psi)
    steps = round(Int, T / 0.5)
    for _ in 1:steps
        phi, _ = exponentiate(x -> H * x, -0.5im, phi; tol=1e-10, ishermitian=true)
    end
    return -log(abs(dot(psi, phi))) / N
end

echo_cache = "../results/data/nb13_krylov_echo.jld2"
rates = isfile(echo_cache) ? load(echo_cache, "rates") : Dict{Tuple{Float64,Int,Float64},Float64}()
for p in (0.0, 0.1, 0.5), N in (12, 16, 20), T in (1.0, 2.0, 3.0)
    haskey(rates, (p, N, T)) && continue
    rates[(p, N, T)] = krylov_rate(N, p, T)
    jldsave(echo_cache; rates)
    @printf("krylov p=%.1f N=%d T=%.0f  rate=%.6f\n", p, N, T, rates[(p, N, T)])
end

# extrapolate in 1/N
using LsqFit
@. lin1N(x, q) = q[1] + q[2] * x
ell_inf = Dict((p, T) => curve_fit(lin1N, [1/12, 1/16, 1/20],
               [rates[(p, N, T)] for N in (12, 16, 20)], [0.4, 0.1]).param[1]
               for p in (0.0, 0.1, 0.5), T in (1.0, 2.0, 3.0))
println("extrapolation done")

extrapolation done


In [4]:
# leading eigenvalue of both columns at the same points, largest modulus (what the echo selects)
mu0_cache = "../results/data/nb13_mu0.jld2"
mu0 = isfile(mu0_cache) ? load(mu0_cache, "mu0") : Dict{Tuple{Float64,Symbol,Float64},ComplexF64}()
# merge anything precomputed by the per-coupling workers
for pc in ("0.0", "0.1", "0.5")
    wf = "../results/data/nb13_mu0_p$(pc).jld2"
    isfile(wf) && merge!(mu0, load(wf, "mu0"))
end
for p in (0.0, 0.1, 0.5), col in (:legacy3, :bulk5), T in (1.0, 2.0, 3.0)
    haskey(mu0, (p, col, T)) && continue
    p == 0.0 && col == :bulk5 && continue        # identical operator at p=0, reuse legacy
    mpo, scaffold = build_alcaraz_tmpo(T; p=p, nbeta=4, column=col)
    theta, _, _, info = block_transfer_eigs(mpo, scaffold; k=2, maxdim=64, cutoff=1e-12,
                                            eigvals_only=true, itermax=300)
    mu0[(p, col, T)] = theta[argmax(abs.(theta))]
    jldsave(mu0_cache; mu0)
    @printf("mu0 p=%.1f %-8s T=%.0f  |mu0|=%.6f  (%s)\n", p, col, T, abs(mu0[(p, col, T)]), info[:reason])
end

println()
@printf("%-5s %-4s %-10s %-13s %-13s\n", "p", "T", "krylov", "legacy dev", "bulk dev")
for p in (0.0, 0.1, 0.5), T in (1.0, 2.0, 3.0)
    ell = ell_inf[(p, T)]
    d3 = -log(abs(mu0[(p, :legacy3, T)])) - ell
    d5 = p == 0.0 ? d3 : -log(abs(mu0[(p, :bulk5, T)])) - ell
    @printf("%-5.1f %-4.0f %-10.5f %+-13.5f %+-13.5f\n", p, T, ell, d3, d5)
end

[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=140|"S=1/2,Site") <-> (dim=2|id=140|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=129|"Link,l=1") <-> (dim=7|id=244|"Link,l=2"), normdiff = 1.1566412865695994
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=506|"S=1/2,Site") <-> (dim=2|id=506|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=450|"Link,l=1") <-> (dim=7|id=227|"Link,l=2"), normdiff = 1.4350175351861996
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


mu0 p=0.1 bulk5    T=1  |mu0|=1.571848  (stuck)


[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=635|"S=1/2,Site") <-> (dim=2|id=635|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=146|"Link,l=1") <-> (dim=7|id=156|"Link,l=2"), normdiff = 1.1566412865695994
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=387|"S=1/2,Site") <-> (dim=2|id=387|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=575|"Link,l=1") <-> (dim=7|id=454|"Link,l=2"), normdiff = 1.4350175351861996
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


...

mu0 p=0.5 bulk5    T=2  |mu0|=1.863922  (stuck)


[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=895|"S=1/2,Site") <-> (dim=2|id=895|"S=1/2,Site")', normdiff = 0.27616420236277306
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=627|"Link,l=1") <-> (dim=7|id=144|"Link,l=2"), normdiff = 1.3391131620216263
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=435|"S=1/2,Site") <-> (dim=2|id=435|"S=1/2,Site")', normdiff = 0.27616420236277306
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=863|"Link,l=1") <->

mu0 p=0.5 bulk5    T=3  |mu0|=1.863856  (maxiter)



p     T    krylov     legacy dev    bulk dev     
0.0   1    -0.41400   +0.00101      +0.00101     


0.0   2    -0.40404   +0.00101      +0.00101     
0.0   3    -0.39925   -0.00103      -0.00103     
0.1   1    -0.45384   +0.00349      +0.00159     
0.1   2    -0.44765   +0.00492      +0.00175     
0.1   3    -0.44963   +0.00949      +0.00545     
0.5   1    -0.62781   +0.03173      +0.00475     
0.5   2    -0.63001   -0.00942      +0.00733     
0.5   3    -0.59935   -0.04702      -0.02330     


## The sector operator on the corrected column

The other line of work found an exact diagonal $Z_2$ operator on the 7-dimensional column, read
from the intertwiner nullspace. If that structure is physical it must exist on the corrected
column too, with a 13-entry signature and a one-dimensional nullspace; `ksector_signs` errors
otherwise.

In [5]:
for p in (0.1, 0.3)
    signs = ksector_signs(p; column=:bulk5)
    @printf("p=%.1f  d=%d  signs = %s\n", p, length(signs), join(Int.(round.(real.(signs))), ","))
end

[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=879|"S=1/2,Site") <-> (dim=2|id=879|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=735|"Link,l=1") <-> (dim=7|id=70|"Link,l=2"), normdiff = 1.1566412865695994
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=281|"S=1/2,Site") <-> (dim=2|id=281|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=448|"Link,l=1") <-> (dim=7|id=967|"Link,l=2"), normdiff = 1.4350175351861996
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


p=0.1  d=13  signs = 1,-1,1,-1,1,-1,1,-1,1,-1,1,-1,1


p=0.3  d=13  signs = -1,1,-1,1,-1,1,-1,1,-1,1,-1,1,-1


[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=363|"S=1/2,Site") <-> (dim=2|id=363|"S=1/2,Site")', normdiff = 0.22953505077375283
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=900|"Link,l=1") <-> (dim=7|id=169|"Link,l=2"), normdiff = 1.222384963003563
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=353|"S=1/2,Site") <-> (dim=2|id=353|"S=1/2,Site")', normdiff = 0.22953505077375283
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=264|"Link,l=1") <-> 

## Production smoke test

One entropy rung through the full production path on the corrected column: block power method,
RTM truncation, generalized R\'enyi-2 profile. The check here is only that the pipeline runs and
returns a finite profile; the physics of the profiles is the subject of the running comparison.

In [6]:
res = compute_entropies(AlcarazParams(lambda=1.0, p=0.1), 2.0;
                        scheme=AlcarazVD2(), nbeta=4, maxdim=32,
                        use_block_pm=true, k_block=2, column=:bulk5)
ok = all(isfinite, res.re) && all(isfinite, res.im)
@printf("profile of %d cuts, all finite: %s\n", length(res.re), ok)
@printf("Re S2 head: %s\n", join(round.(res.re[1:min(5, end)], digits=4), "  "))

[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=268|"S=1/2,Site") <-> (dim=2|id=268|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=704|"Link,l=1") <-> (dim=7|id=456|"Link,l=2"), normdiff = 1.1566412865695994
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on physical(space) => bond(time) indices
┌ Warning: Tensor *not* symmetric (dim=2|id=592|"S=1/2,Site") <-> (dim=2|id=592|"S=1/2,Site")', normdiff = 0.2034249757191385
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93
[ Info: Checking symmetry MPO tensor on bond(space) => phys(time) indices
┌ Warning: Tensor *not* symmetric (dim=7|id=495|"Link,l=1") <-> (dim=7|id=24|"Link,l=2"), normdiff = 1.4350175351861996
└ @ ITransverse ~/.julia/packages/ITransverse/8pmYI/src/ITenUtils/itensor_utils.jl:93


profile of 23 cuts, all finite: true
Re S2 head: 0.0521  0.0924  0.0953  0.1056  0.1213
